In [1]:
import pandas as pd
import numpy as np
import scanpy as sc
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from scipy.spatial.distance import cdist
import torch.nn.functional as F


In [32]:
adata = sc.read_h5ad('../example.h5ad')

step1. ensuring the spot x,y are absoulte distance


In [3]:
#for visium data
adata.obs['X'] = adata.obs['array_row'] *100
adata.obs['Y'] = adata.obs['array_col'] *100
adata.obs

,in_tissue,array_row,array_col,n_counts,X,Y
AAACAAGTATCTCCCA-1,1,50,102,5625.0,5000,10200
AAACACCAATAACTGC-1,1,59,19,9159.0,5900,1900
AAACAGAGCGACTCCT-1,1,14,94,2660.0,1400,9400
AAACAGGGTCTATATT-1,1,47,13,9020.0,4700,1300
AAACAGTGTTCCTGGG-1,1,73,43,9714.0,7300,4300
...,...,...,...,...,...,...
TTGTTGTGTGTCAAGA-1,1,31,77,7260.0,3100,7700
TTGTTTCACATCCAGG-1,1,58,42,7510.0,5800,4200
TTGTTTCATTAGTCTA-1,1,60,30,5742.0,6000,3000
TTGTTTCCATACAACT-1,1,45,27,20118.0,4500,2700


In [34]:
adata.obs

,in_tissue,array_row,array_col,n_counts,X,Y,tcr,cell_type
AAACAAGTATCTCCCA-1,1,50,102,5625.0,5000,10200,1,1
AAACACCAATAACTGC-1,1,59,19,9159.0,5900,1900,1,0
AAACAGAGCGACTCCT-1,1,14,94,2660.0,1400,9400,0,1
AAACAGGGTCTATATT-1,1,47,13,9020.0,4700,1300,1,1
AAACAGTGTTCCTGGG-1,1,73,43,9714.0,7300,4300,0,0
...,...,...,...,...,...,...,...,...
TTGTTGTGTGTCAAGA-1,1,31,77,7260.0,3100,7700,0,1
TTGTTTCACATCCAGG-1,1,58,42,7510.0,5800,4200,1,1
TTGTTTCATTAGTCTA-1,1,60,30,5742.0,6000,3000,1,1
TTGTTTCCATACAACT-1,1,45,27,20118.0,4500,2700,0,1


In [35]:
adata.write_h5ad('../example.h5ad')

step2 if there is not cell type in the h5ad intergrate cell type

In [12]:
cell_type = pd.read_csv('../annotation.txt', sep='\t',index_col=0)
cell_type


,cell_type,cluster,X,Y,T_mean
ID,,,,,
s_016um_00052_00082-1,Non-Tumor,10,5307.678917,12449.199920,-0.173486
s_016um_00010_00367-1,Tumor,0,6040.988191,16999.787109,-0.115027
s_016um_00163_00399-1,Non-Tumor,23,3599.047921,17543.405848,1.293282
s_016um_00238_00388-1,Non-Tumor,21,2396.768054,17382.861868,-0.173486
s_016um_00144_00175-1,Non-Tumor,1,3855.484646,13956.137643,-0.091925
...,...,...,...,...,...
s_016um_00375_00231-1,Non-Tumor,9,172.852112,14900.346458,-0.173486
s_016um_00109_00223-1,Non-Tumor,18,4425.694211,14716.500350,-0.088819
s_016um_00039_00175-1,Non-Tumor,12,5535.624672,13933.920353,-0.173486


In [13]:
cell_type['cell_type'] = [1 if x == 'Tumor' else 2 if x == 'T cell' else 0 for x in cell_type['cell_type']]
cell_type

,cell_type,cluster,X,Y,T_mean
ID,,,,,
s_016um_00052_00082-1,0,10,5307.678917,12449.199920,-0.173486
s_016um_00010_00367-1,1,0,6040.988191,16999.787109,-0.115027
s_016um_00163_00399-1,0,23,3599.047921,17543.405848,1.293282
s_016um_00238_00388-1,0,21,2396.768054,17382.861868,-0.173486
s_016um_00144_00175-1,0,1,3855.484646,13956.137643,-0.091925
...,...,...,...,...,...
s_016um_00375_00231-1,0,9,172.852112,14900.346458,-0.173486
s_016um_00109_00223-1,0,18,4425.694211,14716.500350,-0.088819
s_016um_00039_00175-1,0,12,5535.624672,13933.920353,-0.173486


In [14]:
common_indices = list(set(adata.obs_names) & set(cell_type.index))
print("Common indices:", len(common_indices))

Common indices: 137051


In [15]:
adata.obs['cell_type'] = cell_type.loc[adata.obs_names, 'cell_type'].values

In [17]:
adata.obs

,tumor_gene_signature,stroma_immune_singnature_genes,t_gene_signature,T_means,leiden,X,Y,cell_type
s_016um_00052_00082-1,-13.163058,-110.718834,-7.980356,-0.173486,10,5307.678916618804,12449.199919631592,0
s_016um_00010_00367-1,-10.281478,-75.933578,-5.291239,-0.115027,0,6040.988190604934,16999.787108627097,1
s_016um_00163_00399-1,8.234170,488.635284,59.490974,1.293282,23,3599.0479211059037,17543.40584821326,0
s_016um_00238_00388-1,-4.404501,-17.279686,-7.980356,-0.173486,21,2396.76805388968,17382.86186754705,0
s_016um_00144_00175-1,4.236984,-45.988926,-4.228531,-0.091925,1,3855.484646416702,13956.137643030437,0
...,...,...,...,...,...,...,...,...
s_016um_00375_00231-1,-3.660754,23.671553,-7.980356,-0.173486,9,172.85211221427807,14900.346458407554,0
s_016um_00109_00223-1,-7.506592,43.515171,-4.085675,-0.088819,18,4425.694210888937,14716.500350242153,0
s_016um_00039_00175-1,-4.674540,-8.500046,-7.980356,-0.173486,12,5535.624672456852,13933.920353056192,0
s_016um_00037_00193-1,-13.163058,-103.328911,-7.980356,-0.173486,10,5571.488924089748,14221.434174253345,0


In [22]:
adata.write_h5ad('../test.h5ad')

step3: slecet top 500 highly expressed gene in tumor region

In [23]:
tumor_cells = adata[adata.obs['cell_type'] == 1]
tumor_cells.X.shape

(65742, 18085)

In [24]:
mean_expression = np.mean(tumor_cells.X, axis=0)
top_500_genes = np.argsort(mean_expression)[-500:]


In [25]:
top_500_genes

array([16415,  5011, 18082, 16582, 14607,  7638, 17266, 17667,  3464,
        1899,  7705, 16728, 16611, 18083,  2462,  7541, 14585,  3938,
       13544,  5749,  4523, 16339, 16618, 14668,  8166,  7815,   626,
        4618,  8323, 12777,  9296,  2784,  3384,  7580,  7690,  7575,
       14999, 15506,  8566,  5313, 16631,  2717,  1322,   851,   534,
        2685,  7742,  2205,  7694, 16853, 10209, 16274,  6747, 16981,
       14302,  1937, 17872, 14437, 16590,   878,  8572,  6890,  5842,
        6042,  3332, 14460, 12195,  9764,  3777, 10377, 12279, 10274,
       13601,  6150, 16371, 15874, 17242, 10296, 11099,  4159, 16350,
        3525,  7538,  6619,  8435, 12051, 12715,  3248, 16418, 15112,
       14306,  5669,  9778, 16687, 14052,  8207, 16303,  6638, 11772,
        6018, 17408,  6204, 10361, 10685,  3674,  9298, 11660, 14239,
       12690,  5902, 13557, 17024, 11920,  3941,  7910,  2552, 16537,
       16289,  8204,  3964,  9332,  3691, 12471,  1216,   612,  5349,
       14640, 13582,

In [26]:
adata = adata[:, top_500_genes]

In [27]:
adata.var_names

Index(['ID1', 'CKMT2', 'MT-ND5', 'DNTTIP1', 'EIF4A3', 'ZFAND1', 'TSPO', 'OGT',
       'SUCLG2', 'ADI1',
       ...
       'PYGB', 'PHGR1', 'ELF3', 'CEACAM6', 'KLF5', 'CD24', 'ST14', 'CEACAM5',
       'MUC12', 'EPCAM'],
      dtype='object', length=500)

In [28]:
adata.X.shape

(137051, 500)

In [29]:
adata.var_names

Index(['ID1', 'CKMT2', 'MT-ND5', 'DNTTIP1', 'EIF4A3', 'ZFAND1', 'TSPO', 'OGT',
       'SUCLG2', 'ADI1',
       ...
       'PYGB', 'PHGR1', 'ELF3', 'CEACAM6', 'KLF5', 'CD24', 'ST14', 'CEACAM5',
       'MUC12', 'EPCAM'],
      dtype='object', length=500)

In [30]:
adata.write_h5ad('../test.h5ad')

In [4]:
data = pd.read_csv('hs38d1.fa_cnvkit.txt', sep='\t',header=None)

In [11]:
human = data[0]

In [14]:
human = human.drop_duplicates()

In [17]:
human.to_csv('human.csv',index=False,header=['Gene'])